In [1]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

In [2]:
# Sample technical documentation text
sample_doc = """
# Architecture Overview
Our system is built using FastAPI and LangGraph for agentic orchestration. 
The backend connects to PostgreSQL for long-term user persistence and uses Redis for caching.

## State Management
State is centralized using TypedDict schemas. Every graph node receives the current state dictionary, 
mutates specific keys, and returns updates via reducers like operator.add.

### Checkpointing
Checkpointers such as MemorySaver allow thread-level session recovery. 
Threads are isolated by unique thread_id configurations.
"""

In [3]:
# 1. Markdown Header Splitter (Preserves document structure)
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(sample_doc)

In [4]:
print(f"--- Markdown Header Splits: {len(md_header_splits)} sections ---")
for split in md_header_splits:
  print(f"Metadata: {split.metadata}")
  print(f"Content Preview: {split.page_content[:60]}...\n")

--- Markdown Header Splits: 3 sections ---
Metadata: {'Header 1': 'Architecture Overview'}
Content Preview: Our system is built using FastAPI and LangGraph for agentic ...

Metadata: {'Header 1': 'Architecture Overview', 'Header 2': 'State Management'}
Content Preview: State is centralized using TypedDict schemas. Every graph no...

Metadata: {'Header 1': 'Architecture Overview', 'Header 2': 'State Management', 'Header 3': 'Checkpointing'}
Content Preview: Checkpointers such as MemorySaver allow thread-level session...



In [5]:
# 2. Recursive Character Text Splitter (Fine-grained token-aware text chunking)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,  # Max characters per chunk
    chunk_overlap=30,  # Overlap to preserve boundary context
    separators=["\n\n", "\n", " ", ""],
)

In [6]:
fine_chunks = text_splitter.split_documents(md_header_splits)

In [7]:
print(
    f"--- Recursive Fine Chunks Generated: {len(fine_chunks)} total chunks ---"
)
for i, chunk in enumerate(fine_chunks):
  print(f"Chunk [{i}] (Length {len(chunk.page_content)}):")
  print(f"  {repr(chunk.page_content)}")
  print(f"  Metadata: {chunk.metadata}\n")

--- Recursive Fine Chunks Generated: 5 total chunks ---
Chunk [0] (Length 74):
  'Our system is built using FastAPI and LangGraph for agentic orchestration.'
  Metadata: {'Header 1': 'Architecture Overview'}

Chunk [1] (Length 93):
  'The backend connects to PostgreSQL for long-term user persistence and uses Redis for caching.'
  Metadata: {'Header 1': 'Architecture Overview'}

Chunk [2] (Length 101):
  'State is centralized using TypedDict schemas. Every graph node receives the current state dictionary,'
  Metadata: {'Header 1': 'Architecture Overview', 'Header 2': 'State Management'}

Chunk [3] (Length 74):
  'mutates specific keys, and returns updates via reducers like operator.add.'
  Metadata: {'Header 1': 'Architecture Overview', 'Header 2': 'State Management'}

Chunk [4] (Length 127):
  'Checkpointers such as MemorySaver allow thread-level session recovery.\nThreads are isolated by unique thread_id configurations.'
  Metadata: {'Header 1': 'Architecture Overview', 'Header 2': 'S

# Advanced Chunking Strategies

1. Semantic Chunking — splits text based on changes in meaning.
2. Recursive Token-Based Chunking — splits text into token-limited chunks while preserving natural boundaries where possible.
3. Parent-Child Document Chunking — retrieves small chunks for accurate search but returns larger parent documents for richer context.

## 1. Semantic Chunking

In [1]:

from langchain_core.documents import Document

text = """
Retrieval Augmented Generation (RAG) is a technique
that combines information retrieval with large
language models to generate grounded answers.

A RAG system retrieves relevant documents from
a vector database. These documents are passed
to the LLM as context for answering questions.

Chunking is an important step in RAG. Large
documents are divided into smaller sections
before generating embeddings.

Semantic chunking groups text by meaning.
It uses embeddings to detect topic changes
between sentences.

Recursive chunking splits text using paragraphs,
sentences, spaces, and token limits.

Parent-child retrieval indexes smaller chunks
but returns larger parent documents to preserve
context during answer generation.

Python is widely used in artificial intelligence.
It supports machine learning libraries such as
PyTorch, TensorFlow, and scikit-learn.

Machine learning models learn statistical
patterns from data to make predictions.
"""

documents = [
    Document(
        page_content=text,
        metadata={"source": "rag_notes.txt"}
    )
]


In [7]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

In [8]:
# Step 1: Initialize embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


c:\Users\User\Desktop\AI Developer\Practical Agentic Practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\User\Desktop\AI Developer\Practical Agentic Practice\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activat

In [9]:
# Step 2: Configure semantic splitter
semantic_splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=80
)

In [10]:
# Step 3: Split documents based on meaning
semantic_chunks = semantic_splitter.split_documents(
    documents
)

In [11]:
# Step 4: Display chunks
for i, chunk in enumerate(semantic_chunks, 1):
    print(f"\n--- Semantic Chunk {i} ---")
    print(chunk.page_content)



--- Semantic Chunk 1 ---

Retrieval Augmented Generation (RAG) is a technique
that combines information retrieval with large
language models to generate grounded answers. A RAG system retrieves relevant documents from
a vector database. These documents are passed
to the LLM as context for answering questions. Chunking is an important step in RAG. Large
documents are divided into smaller sections
before generating embeddings.

--- Semantic Chunk 2 ---
Semantic chunking groups text by meaning. It uses embeddings to detect topic changes
between sentences. Recursive chunking splits text using paragraphs,
sentences, spaces, and token limits. Parent-child retrieval indexes smaller chunks
but returns larger parent documents to preserve
context during answer generation. Python is widely used in artificial intelligence.

--- Semantic Chunk 3 ---
It supports machine learning libraries such as
PyTorch, TensorFlow, and scikit-learn. Machine learning models learn statistical
patterns from data to 

## 2.Recursive Token-Based Chunking

In [12]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)
import tiktoken
